# Preprocessing the CERT Dataset for Model Training:

### Imports:

In [1]:
import os
import numpy as np
import pandas as pd
from urllib.parse import urlparse

## **Layer A**

### Defining Our Constants:

In [2]:
DEFAULT_OUTPUT_DIR = "processed_datasets"
WORK_HOURS = (9, 17)
INTERNAL_EMAIL_DOMAIN = "dtaa.com"
LONG_URL_THRESHOLD = 80

JOB_DOMAINS = {
    "careerbuilder.com",
    "indeed.com",
    "monster.com",
    "simplyhired.com",
    "linkedin.com",
    "www.linkedin.com"
}

CLOUD_STORAGE_DOMAINS = {
    "dropbox.com",
    "www.dropbox.com",
    "drive.google.com",
    "docs.google.com",
    "yousendit.com",
    "www.yousendit.com"
}

SUSPICIOUS_DOMAINS = {
    "wikileaks.org",
    "www.wikileaks.org"
}

### Importing the CERT Dataset:

In [3]:
def load_raw_logs(cert_path: str) -> dict:
    """
    Loads the CERT log files needed for preprocessing.
    
    Args:
        cert_path: The base path containing the CERT dataset
        
    Returns:
        dict: A mapping of the file name to a raw Dataframe: {file_name: DataFrame}
    """
    file_map = {
        "logon": "logon.csv",
        "file": "file.csv",
        "device": "device.csv",
        "email": "email.csv",
        "http": "http.csv"
    }
    
    logs = {}
    missing_files = []
    
    for source_name, filename in file_map.items():
        full_path = os.path.join(cert_path, filename)
        if os.path.exists(full_path):
            logs[source_name] = pd.read_csv(full_path)
        else:
            missing_files.append(filename)
            
    if missing_files:
        raise FileNotFoundError("Missing required CERT files: " + ", ".join(missing_files))
    
    return logs

In [4]:
cert_path = r"C:\Users\loera\Documents\Datasets\CERT_r6.2"

In [ ]:
logs = load_raw_logs(cert_path)

In [5]:
logons = pd.read_csv(os.path.join(cert_path, "logon.csv"))

In [ ]:
# logons.head()

In [6]:
files = pd.read_csv(os.path.join(cert_path, "file.csv"))

In [ ]:
files.head()

In [7]:
devices = pd.read_csv(os.path.join(cert_path, "device.csv"))

In [ ]:
devices.head()

In [8]:
emails = pd.read_csv(os.path.join(cert_path, "email.csv"))

KeyboardInterrupt: 

In [ ]:
emails.head()

In [ ]:
http = pd.read_csv(os.path.join(cert_path, "http.csv"))

### Normalizing Data Within Shared Columns:

In [ ]:
def normalize_shared_columns(df: pd.DataFrame, remove_cols: list=["id"]) -> pd.DataFrame:
    """
    Normalizes CERT log files across commonly shared columns. Additionally drops columns that are deemed irrelevant.
    
    Args:
        df: The raw CERT dataframe (logon, file, device, or email)
        remove_cols: The columns to drop from the original CERT file
        
    Returns:
        pd.Dataframe: A normalized dataframe with consistent identifiers and fields
    """
    # Standardizing column names
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    
    # Renaming data column
    if "date" in df.columns:
        df.rename(columns={"date": "timestamp"}, inplace=True)
        
    if "timestamp" not in df.columns:
        raise KeyError("Expected a 'date' or 'timestamp' column in DataFrame.")
        
    # Converting timestamp to datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    
    # Dropping rows with invalid timestamps
    df.dropna(axis=0, subset=["timestamp"], inplace=True)
    
    # Creating a 'day' aggregation key column
    df["day"] = df["timestamp"].dt.floor("D")
    
    # Normalizing identifiers
    df["user"] = df["user"].astype(str).str.lower().str.strip()
    df["pc"] = df["pc"].astype(str).str.lower().str.strip()
    
    # Dropping unusable columns
    remove_cols = [col.lower().strip() for col in remove_cols]
    cols_to_drop = [col for col in remove_cols if col in df.columns]
    df.drop(axis=1, columns=cols_to_drop, inplace=True)
    
    # Sorting rows for consistency
    df.sort_values(by=["user", "pc", "timestamp"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    return df

In [ ]:
norm_logons = normalize_shared_columns(logons)

In [ ]:
norm_logons.head()

In [ ]:
norm_files = normalize_shared_columns(files)

In [ ]:
norm_files.head()

In [ ]:
norm_devices = normalize_shared_columns(devices)

In [ ]:
norm_devices.head()

In [ ]:
norm_emails = normalize_shared_columns(emails)

In [ ]:
norm_emails.head()

In [ ]:
# Sanity checks
files = [norm_logons, norm_files, norm_devices, norm_emails]

for file in files:
    assert file["day"].dtype == "datetime64[ns]"
    assert set(["user", "pc", "timestamp", "day"]).issubset(norm_logons.columns)

### Feature Engineering Normalized Data:

In [ ]:
def extract_logon_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily authentication behavior features from logon events to create an aggregated feature table.
    
    Args:
        norm_df: The normalized logon dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated logon behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)    
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving logon-related features
    features = grouped.agg(
        logon_count = ("activity", lambda x: (x=="Logon").sum()),
        logoff_count = ("activity", lambda x: (x=="Logoff").sum()),
        off_hours_logon = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [ ]:
agg_logons = extract_logon_features(norm_logons)

In [ ]:
agg_logons.head()

In [ ]:
def extract_file_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily file access behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated file behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving file-related features
    features = grouped.agg(
        file_open_count = ("activity", lambda x: (x=="File Open").sum()),
        file_write_count = ("activity", lambda x: (x=="File Write").sum()),
        file_copy_count = ("activity", lambda x: (x=="File Copy").sum()),
        file_delete_count = ("activity", lambda x: (x=="File Delete").sum()),
        unique_files_accessed = ("filename", "nunique"),
        off_hours_files_accessed = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [ ]:
agg_files = extract_file_features(norm_files)

In [ ]:
agg_files.head()

In [ ]:
def extract_device_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily removable media (USB) behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated removable media behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving media-related features
    features = grouped.agg(
        usb_insert_count = ("activity", lambda x: (x=="Connect").sum()),
        usb_remove_count = ("activity", lambda x: (x=="Disconnect").sum()),
        off_hours_usb_usage = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [ ]:
agg_devices = extract_device_features(norm_devices)

In [ ]:
agg_devices.head()

In [ ]:
def extract_email_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily email communication behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated email behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # External email heuristic
    df["external_email"] = ~df["to"].str.contains("dtaa.com", na=False)
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving email-related features
    features = grouped.agg(
        emails_sent = ("to", "count"),
        unique_recipients = ("to", "nunique"),
        external_emails = ("external_email", "sum"),
        attachments_sent = ("attachments", lambda x: (x.notnull()).sum()),
        off_hours_emails = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [ ]:
agg_emails = extract_email_features(norm_emails)

In [ ]:
agg_emails.head()

### Merging Aggregated Feature Tables Into One Behavioral Matrix:

In [ ]:
def merge_behavioral_features(feature_tables: list[pd.DataFrame], merge_cols: list=["user", "pc", "day"]) -> pd.DataFrame:
    """
    Merges multiple aggregated behavioral feature tables into a single dataset.
    
    Args:
        feature_tables: A list of aggregated features dataframes to merge, where each table must contain the merge columns
        merge_cols: columns to merge on
        
    Returns:
        pd.DataFrame: A unified table with missing activity filled with zeros
    """
    # Filtering out empty tables
    valid_tables = [df for df in feature_tables if ((df is not None) and (not df.empty))]
    
    if not valid_tables:
        raise ValueError("No valid feature tables provided for merging")

    merged_df = valid_tables[0].copy()
    
    # Iteratively merging tables
    for df in valid_tables[1:]:
        merged_df = merged_df.merge(df, on=merge_cols, how="outer")
        
    # Identifying feature columns, excluding identifiers
    feature_cols = [col for col in merged_df.columns if col not in merge_cols]
    
    # Filling missing feature values with zero
    merged_df[feature_cols] = merged_df[feature_cols].fillna(0)
    merged_df[feature_cols] = merged_df[feature_cols].astype("int32")
    
    # Sorting dataframe for consistency
    merged_df.sort_values(by=merge_cols, inplace=True)
    merged_df.reset_index(drop=True, inplace=True)
    
    return merged_df

In [ ]:
feature_tables = [agg_logons, agg_files, agg_devices, agg_emails]

In [ ]:
behavioral_matrix = merge_behavioral_features(feature_tables)

In [ ]:
behavioral_matrix

In [ ]:
# Ensuring there are no duplicate rows
assert behavioral_matrix.duplicated(subset=["user", "pc", "day"]).sum() == 0

# Ensuring there are no empty cells in the feature space
assert behavioral_matrix.drop(columns=["user", "pc", "day"]).isna().sum().sum() == 0

### Feature Engineering PC-Related Data:

In [ ]:
def extract_pc_features(df: pd.DataFrame, min_history: int=5) -> pd.DataFrame:
    """
    Extracts PC-derived contextual features to an aggregated behavioral matrix.
    
    Args:
        min_history: Minimum prior observations required before flagging new PC usage as abnormal activity
        
    Returns:
        pd.DataFrame: A behavioral matrix with added user-to-PC behavioral history
    """
    df = df.copy()
    df.sort_values(by=["user", "day", "pc"], inplace=True)
    
    # Tracking historical PC usage counts
    df["pc_prior_use_count"] = df.groupby(["user", "pc"]).cumcount()
    df["user_total_prior_days"] = df.groupby("user").cumcount()
    df["pc_seen_before"]  = (df["pc_prior_use_count"] > 0).astype(int)
    
    df["pc_prior_use_ratio"] = np.where(
        df["user_total_prior_days"] > 0,
        df["pc_prior_use_count"] / df["user_total_prior_days"],
        0.0
    )
    
    # Identifying a user's primary PC (i.e., most-used PC)
    primary_pc_map = (
        df.groupby(["user", "pc"])
        .size()
        .reset_index(name="count")
        .sort_values(["user", "count", "pc"], ascending=[True, False, True])
        .drop_duplicates(subset=["user"])
        .set_index("user")["pc"]
        .to_dict()
    )
    
    df["pc_is_primary"] = df.apply(
        lambda row: int(row["pc"] == primary_pc_map.get(row["user"])),
        axis=1
    )
    
    # Tracking the number of distinct PC's previously used
    df["distinct_pcs_prior"] = (
        df.groupby(["user"])["pc"]
        .transform(lambda s: [len(set(s.iloc[:i])) for i in range(len(s))])
    )
    
    # Tracking the number of unique PC's used on a given day
    same_day_counts = (
        df.groupby(["user", "day"])["pc"]
        .transform("nunique")
    )
    
    df["multi_pc_day"] = same_day_counts
    
    # Identifies new PC usage after an established history
    df["new_pc_after_history"] = (
        (df["pc_prior_use_count"] == 0) &
        (df["user_total_prior_days"] >= min_history)
    ).astype(int)
    
    df.drop(columns=["user_total_prior_days"], inplace=True)
    
    return df

In [ ]:
pc_behavioral_matrix = extract_pc_features(behavioral_matrix)

In [ ]:
pc_behavioral_matrix.head()

### Integrating PC and UEBA-Specific Enhancements:

In [ ]:
def apply_ueba_enhancements(df: pd.DataFrame, id_cols: list=["user", "pc", "day"], rolling_window: int=7) -> pd.DataFrame:
    """
    Applying UEBA-specific enhancements to a behavioral matrix such as:
    - Per-user causal z-score deviations based only on prior history
    - Causal rolling mean deltas that exclude the current row
    - Cross-channel risk flags
    
    Args:
        df: A merged behavioral matrix
        id_cols: Identifier columns to exclude from feature calculation
        rolling_window: Window size in days for rolling statistics
        
    Returns:
        pd.DataFrame: An enhanced UEBA-ready feature matrix
    """
    df = df.copy()
    df.sort_values(by=["user", "day"], inplace=True)
    
    feature_cols = [col for col in df.columns if col not in id_cols]
    
    # Defining per-user causal z-score deviations
    for col in feature_cols:
        grouped = df.groupby("user")[col]
        
        # Building baselines from prior observations
        prior_mean = grouped.transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
        prior_std = grouped.transform(lambda x: x.shift(1).expanding(min_periods=2).std())
        
        # Avoiding division by zero
        prior_std.replace(0, np.nan, inplace=True)
        
        # Computing z-scores
        df[f"{col}_zscore"] = ((df[col] - prior_mean) / prior_std).fillna(0.0)
        
    # Defining causal rolling temporal deltas
    for col in feature_cols:
        prior_rolling_mean = (
            df.groupby("user")[col]
            .transform(lambda x: x.shift(1).rolling(window=rolling_window, min_periods=1).mean())
        )
        
        # First observations have no prior window, so the delta defaults to 0.
        df[f"{col}_rolling_delta"] = (df[col] - prior_rolling_mean).fillna(0.0)
        
    # Defining cross-channel risk flags
    df["usb_file_activity_flag"] = ((df.get("usb_insert_count", 0) > 0) & (df.get("file_write_count", 0) > 0)).astype(int)
    
    df["off_hours_activity_flag"] = ((df.filter(like="off_hours").sum(axis=1) > 0)).astype(int)
    
    df["external_comm_activity_flag"] = (df.get("external_emails", 0) > 0).astype(int)
    
    df["non_primary_pc_usb_flag"] = ((df["pc_is_primary"] == 0) & (df.get("usb_insert_count", 0) > 0)).astype(int)
    
    df["non_primary_pc_file_copy_flag"] = ((df["pc_is_primary"] == 0) & (df.get("file_copy_count", 0) > 0)).astype(int)
    
    df["non_primary_pc_off_hours_flag"] = ((df["pc_is_primary"] == 0) & (df.filter(like="off_hours").sum(axis=1) > 0)).astype(int)

    return df

In [ ]:
ueba_matrix = apply_ueba_enhancements(pc_behavioral_matrix)

In [ ]:
ueba_matrix.head()

### Saving the Final UEBA-Enhanced Dataset:

In [ ]:
def save_dataset(dataset: pd.DataFrame, filename: str="ueba_dataset.csv") -> None:
    """
    Saves the final UEBA-enhanced dataset to the specified path as a CSV file.
    
    Args:
        dataset: The UEBA-enhanced dataset
        file_name: The desired name of the CSV dataset
        
    Returns:
        None:
    """
    # Ensures directory exists
    save_path = os.path.join(os.getcwd(), "processed_datasets")
    os.makedirs(save_path, exist_ok=True)
    
    # Creates full file path
    file_path = os.path.join(save_path, filename)
    
    # Saving the dataset
    dataset.to_csv(file_path)
    print(f"Dataset successfully saved to: {file_path}")

In [ ]:
save_dataset(ueba_matrix, "ueba_dataset_2.csv")